# odmlib v0.2.0: Dynamic OID Ref/Def Checking

In CDISC ODM documents, OIDs (Object Identifiers) are the glue that ties metadata together. An `ItemRef` points to an `ItemDef` via `ItemOID`, a `CodeListRef` points to a `CodeList` via `CodeListOID`, and so on. If any of these references are broken. pointing to an OID that doesn't exist, or pointing to the wrong type of element. the document is invalid.

odmlib v0.2.0 introduces **dynamic OID ref/def checking** via `create_oid_checker()`. This replaces the manually-coded `OIDRef` classes from earlier versions with a system that **introspects the model classes at runtime** to automatically discover:

- Which elements define OIDs (e.g., `ItemDef`, `CodeList`, `MethodDef`)
- Which attributes are OID references (e.g., `ItemOID`, `CodeListOID`, `MethodOID`)
- The mapping between each reference attribute and its expected definition element

This means the checker stays in sync with the model automatically — no manual maintenance required, and no risk of the checker falling out of date when the model evolves.

This notebook covers:

1. **Basic usage**: creating a checker and validating OID integrity
2. **What the checker validates**: duplicate OIDs, missing definitions, type mismatches
3. **Finding unreferenced OIDs**: detecting orphaned metadata
4. **Inspecting the discovered mappings**: understanding what the introspection found
5. **Cross-model support**: using the same API for ODM 1.3.2, Define-XML 2.1, and more
6. **Migration from the legacy API**: replacing manual `OIDRef` classes

## Setup

In [1]:
import odmlib.odm_1_3_2.model as ODM
import odmlib.define_2_1.model as DEFINE

from odmlib import create_oid_checker, OdmlibOIDError

## 1. Basic Usage

The workflow is straightforward: create a checker for your model, then call `verify_oids()` on any odmlib object. The checker traverses the entire element tree, registers every OID definition it finds, collects every OID reference, and then validates that every reference points to a valid definition.

Let's start by building a small but complete ODM 1.3.2 MetaDataVersion where all OID references are satisfied.

In [2]:
# Build a small but internally consistent MetaDataVersion.
# The Protocol references a StudyEventDef, which references a FormDef,
# which references an ItemGroupDef, which references ItemDefs.

protocol = ODM.Protocol()
protocol.StudyEventRef = [
    ODM.StudyEventRef(StudyEventOID="SE.BASELINE", OrderNumber=1, Mandatory="Yes")
]

se_baseline = ODM.StudyEventDef(OID="SE.BASELINE", Name="Baseline Visit", Repeating="No", Type="Scheduled")
se_baseline.FormRef = [ODM.FormRef(FormOID="F.DM", Mandatory="Yes", OrderNumber=1)]

form_dm = ODM.FormDef(OID="F.DM", Name="Demographics", Repeating="No")
form_dm.ItemGroupRef = [ODM.ItemGroupRef(ItemGroupOID="IG.DM", Mandatory="Yes")]

igd_dm = ODM.ItemGroupDef(OID="IG.DM", Name="Demographics", Repeating="No")
igd_dm.ItemRef = [
    ODM.ItemRef(ItemOID="IT.SUBJID", Mandatory="Yes", OrderNumber=1),
    ODM.ItemRef(ItemOID="IT.AGE", Mandatory="Yes", OrderNumber=2),
    ODM.ItemRef(ItemOID="IT.SEX", Mandatory="Yes", OrderNumber=3),
]

item_subjid = ODM.ItemDef(OID="IT.SUBJID", Name="Subject ID", DataType="text", Length=20)
item_age = ODM.ItemDef(OID="IT.AGE", Name="Age", DataType="integer", Length=3)
item_sex = ODM.ItemDef(OID="IT.SEX", Name="Sex", DataType="text", Length=1)
item_sex.CodeListRef = ODM.CodeListRef(CodeListOID="CL.SEX")

cl_sex = ODM.CodeList(OID="CL.SEX", Name="Sex", DataType="text")
cl_sex.CodeListItem = [
    ODM.CodeListItem(CodedValue="M", Decode=ODM.Decode(
        TranslatedText=[ODM.TranslatedText(_content="Male", lang="en")])),
    ODM.CodeListItem(CodedValue="F", Decode=ODM.Decode(
        TranslatedText=[ODM.TranslatedText(_content="Female", lang="en")])),
]

mdv = ODM.MetaDataVersion(OID="MDV.001", Name="Demographics v1")
mdv.Protocol = protocol
mdv.StudyEventDef = [se_baseline]
mdv.FormDef = [form_dm]
mdv.ItemGroupDef = [igd_dm]
mdv.ItemDef = [item_subjid, item_age, item_sex]
mdv.CodeList = [cl_sex]

print("MetaDataVersion built with:")
print(f"  1 StudyEventDef  -> references FormDef F.DM")
print(f"  1 FormDef        -> references ItemGroupDef IG.DM")
print(f"  1 ItemGroupDef   -> references 3 ItemDefs")
print(f"  3 ItemDefs       -> IT.SEX references CodeList CL.SEX")
print(f"  1 CodeList")

MetaDataVersion built with:
  1 StudyEventDef  -> references FormDef F.DM
  1 FormDef        -> references ItemGroupDef IG.DM
  1 ItemGroupDef   -> references 3 ItemDefs
  3 ItemDefs       -> IT.SEX references CodeList CL.SEX
  1 CodeList


In [3]:
# Create a checker and validate
checker = create_oid_checker("odm_1_3_2")
result = mdv.verify_oids(checker)

print(f"OID validation passed: {result}")
print(f"\nOID definitions found: {len(checker.oid)}")
for oid, element_type in sorted(checker.oid.items()):
    print(f"  {oid:20s} defined by {element_type}")

OID validation passed: True

OID definitions found: 8
  CL.SEX               defined by CodeList
  F.DM                 defined by FormDef
  IG.DM                defined by ItemGroupDef
  IT.AGE               defined by ItemDef
  IT.SEX               defined by ItemDef
  IT.SUBJID            defined by ItemDef
  MDV.001              defined by MetaDataVersion
  SE.BASELINE          defined by StudyEventDef


The checker traversed the entire element tree, registered 8 OID definitions (the `MetaDataVersion` itself plus its children), collected all the OID references from `Ref` elements, and confirmed that every reference points to a valid definition of the correct type.

After calling `verify_oids()`, you can also inspect what OID references were collected:

In [4]:
# Inspect the collected OID references
print("OID references collected:")
for attr, oid_set in sorted(checker.oid_ref.items()):
    if oid_set:  # only show attributes that had references
        expected_def = checker.ref_def.get(attr, "?")
        print(f"  {attr:25s} -> {sorted(oid_set)}  (expects {expected_def})")

OID references collected:
  CodeListOID               -> ['CL.SEX']  (expects CodeList)
  FormOID                   -> ['F.DM']  (expects FormDef)
  ItemGroupOID              -> ['IG.DM']  (expects ItemGroupDef)
  ItemOID                   -> ['IT.AGE', 'IT.SEX', 'IT.SUBJID']  (expects ItemDef)
  StudyEventOID             -> ['SE.BASELINE']  (expects StudyEventDef)


## 2. What the Checker Validates

The checker catches three categories of problems. Let's trigger each one.

### 2a. Duplicate OIDs

Every OID must be unique within its scope. If two elements share the same OID, the checker raises `OdmlibOIDError` immediately when it encounters the duplicate during traversal.

In [5]:
# Add a second ItemDef with the same OID as an existing one
duplicate_item = ODM.ItemDef(OID="IT.AGE", Name="Age Duplicate", DataType="integer", Length=3)
mdv.ItemDef.append(duplicate_item)

checker_dup = create_oid_checker("odm_1_3_2")
try:
    mdv.verify_oids(checker_dup)
except OdmlibOIDError as e:
    print(f"Caught: {type(e).__name__}")
    print(f"  Message:   {e}")
    print(f"  Attribute: {e.attribute}")
    print(f"  Hint:      {e.hint}")

# Clean up: remove the duplicate so subsequent examples work
mdv.ItemDef.remove(duplicate_item)

Caught: OdmlibOIDError
  Message:   OID IT.AGE is not unique - element ItemDef
  Hint: Each OID must be unique within a MetaDataVersion. OID 'IT.AGE' is already defined in a ItemDef element.
  Attribute: OID
  Hint:      Each OID must be unique within a MetaDataVersion. OID 'IT.AGE' is already defined in a ItemDef element.


### 2b. Missing OID Definitions (Broken References)

If an OID reference points to a definition that doesn't exist, the checker raises `OdmlibOIDError` during the `check_oid_refs()` phase (which runs automatically at the end of `verify_oids()`).

In [6]:
# Add an ItemRef that points to a non-existent ItemDef
igd_dm.ItemRef.append(
    ODM.ItemRef(ItemOID="IT.NONEXISTENT", Mandatory="No", OrderNumber=4)
)

checker_missing = create_oid_checker("odm_1_3_2")
try:
    mdv.verify_oids(checker_missing)
except OdmlibOIDError as e:
    print(f"Caught: {type(e).__name__}")
    print(f"  Message:   {e}")
    print(f"  Attribute: {e.attribute}")
    print(f"  Hint:      {e.hint}")

# Clean up
igd_dm.ItemRef.pop()

Caught: OdmlibOIDError
  Message:   OID IT.NONEXISTENT referenced in attribute ItemOID is not found.
  Hint: Define an element with OID 'IT.NONEXISTENT' before referencing it via ItemOID.
  Attribute: ItemOID
  Hint:      Define an element with OID 'IT.NONEXISTENT' before referencing it via ItemOID.


ItemRef(ItemOID, OrderNumber, Mandatory, KeySequence, MethodOID, Role, RoleCodeListOID, CollectionExceptionConditionOID)

### 2c. OID Type Mismatches

The checker also validates that a reference points to the correct *type* of definition. For example, an `ItemOID` attribute should point to an `ItemDef`, not to a `CodeList`. If the definition exists but has the wrong element type, the checker catches the mismatch.

In [7]:
# Add an ItemRef whose ItemOID points to the CodeList OID instead of an ItemDef.
# The OID "CL.SEX" exists, but it belongs to a CodeList, not an ItemDef.
igd_dm.ItemRef.append(
    ODM.ItemRef(ItemOID="CL.SEX", Mandatory="No", OrderNumber=4)
)

checker_mismatch = create_oid_checker("odm_1_3_2")
try:
    mdv.verify_oids(checker_mismatch)
except OdmlibOIDError as e:
    print(f"Caught: {type(e).__name__}")
    print(f"  Message:   {e}")
    print(f"  Attribute: {e.attribute}")
    print(f"  Hint:      {e.hint}")

# Clean up
igd_dm.ItemRef.pop()

Caught: OdmlibOIDError
  Message:   OID reference for attribute ItemOID element types do not match: ItemDef and CodeList
  Hint: Attribute 'ItemOID' should reference a ItemDef, but OID 'CL.SEX' is defined on a CodeList.
  Attribute: ItemOID
  Hint:      Attribute 'ItemOID' should reference a ItemDef, but OID 'CL.SEX' is defined on a CodeList.


ItemRef(ItemOID, OrderNumber, Mandatory, KeySequence, MethodOID, Role, RoleCodeListOID, CollectionExceptionConditionOID)

### Structured Error Attributes

All three error types raise `OdmlibOIDError`, which is a subclass of `OdmlibValidationError`. Every instance carries structured attributes (`attribute`, `hint`, `element_type`, `actual_value`) that you can use for programmatic error reporting. Since `OdmlibOIDError` also inherits from `ValueError` (for backward compatibility through v0.2.x), existing `except ValueError` handlers will still catch these errors.

## 3. Finding Unreferenced OIDs

Beyond checking that references are valid, you can also find **orphaned definitions**, elements that define an OID but are never referenced by anything. These are valid ODM, but they represent unused metadata that may indicate an authoring mistake or stale leftovers from a previous revision.

In [8]:
# Add an ItemDef that nothing references
orphan_item = ODM.ItemDef(OID="IT.UNUSED_WEIGHT", Name="Weight", DataType="float", Length=5)
mdv.ItemDef.append(orphan_item)

# Add a CodeList that nothing references
orphan_cl = ODM.CodeList(OID="CL.UNUSED_NY", Name="No Yes", DataType="text")
orphan_cl.EnumeratedItem = [
    ODM.EnumeratedItem(CodedValue="N"),
    ODM.EnumeratedItem(CodedValue="Y"),
]
mdv.CodeList.append(orphan_cl)

# Verify OIDs first (required before checking unreferenced)
checker_orphan = create_oid_checker("odm_1_3_2")
mdv.verify_oids(checker_orphan)

# Now find unreferenced OIDs
orphans = mdv.unreferenced_oids(checker_orphan)

if orphans:
    print(f"Found {len(orphans)} unreferenced OID(s):")
    for oid, ref_attr in orphans.items():
        # ref_attr tells us what kind of reference attribute should have pointed here
        print(f"  {oid:25s} (expected reference via {ref_attr})")
else:
    print("All OIDs are referenced.")

# Clean up
mdv.ItemDef.remove(orphan_item)
mdv.CodeList.remove(orphan_cl)

Found 3 unreferenced OID(s):
  MDV.001                   (expected reference via MetaDataVersionOID)
  IT.UNUSED_WEIGHT          (expected reference via ItemOID)
  CL.UNUSED_NY              (expected reference via CodeListOID)


The `unreferenced_oids()` method returns a dictionary mapping each orphaned OID value to the reference attribute name that *should* have pointed to it. This tells you not just that something is unused, but what kind of reference is missing, for example `IT.UNUSED_WEIGHT` should have been referenced via an `ItemOID` attribute in an `ItemRef` element.

Note that `unreferenced_oids()` will call `verify_oids()` automatically if it hasn't been called yet, so you can use it as a single call if you don't need the separate validation step.

## 4. Inspecting the Discovered Mappings

The dynamic checker introspects the model classes to build its mappings. You can inspect what it discovered — this is useful for understanding what the checker will validate and for debugging unexpected results.

The introspection works in four steps:

1. **Discover model classes**: find all `ODMElement` subclasses in the model module
2. **Discover OID definitions**: find classes that have an `OID` attribute (e.g., `ItemDef`, `CodeList`)
3. **Discover OID references**: find attributes containing `"OID"` in their name but not named exactly `"OID"` (e.g., `ItemOID`, `CodeListOID`)
4. **Build ref/def mapping**: determine which reference attribute points to which definition class, using a naming convention (strip `"OID"`, try `<base>Def` then `<base>`) with overrides for irregular names

In [9]:
# Create a fresh checker and inspect what the introspection discovered
checker = create_oid_checker("odm_1_3_2")

print("=== OID Definition Elements ===")
print(f"Classes with an OID attribute ({len(checker.oid_defs)}):")
for cls_name in sorted(checker.oid_defs):
    print(f"  {cls_name}")

print(f"\n=== Ref -> Def Mapping ({len(checker.ref_def)} entries) ===")
print("Which definition class does each reference attribute point to?")
for attr, def_class in sorted(checker.ref_def.items()):
    print(f"  {attr:40s} -> {def_class}")

print(f"\n=== Def -> Ref Mapping ({len(checker.def_ref)} entries) ===")
print("Which reference attributes point to each definition class?")
for def_class, ref_attrs in sorted(checker.def_ref.items()):
    print(f"  {def_class:25s} <- {ref_attrs}")

=== OID Definition Elements ===
Classes with an OID attribute (15):
  ArchiveLayout
  CodeList
  ConditionDef
  FormDef
  ItemDef
  ItemGroupDef
  Location
  MeasurementUnit
  MetaDataVersion
  MethodDef
  Presentation
  SignatureDef
  Study
  StudyEventDef
  User

=== Ref -> Def Mapping (15 entries) ===
Which definition class does each reference attribute point to?
  CodeListOID                              -> CodeList
  CollectionExceptionConditionOID          -> ConditionDef
  FormOID                                  -> FormDef
  ItemGroupOID                             -> ItemGroupDef
  ItemOID                                  -> ItemDef
  LocationOID                              -> Location
  MeasurementUnitOID                       -> MeasurementUnit
  MetaDataVersionOID                       -> MetaDataVersion
  MethodOID                                -> MethodDef
  PresentationOID                          -> Presentation
  RoleCodeListOID                          -> CodeList
 

### Skip Lists

Not every OID-containing attribute is a metadata reference. `FileOID` and `PriorFileOID` are file-level identifiers, not references to definition elements. The checker uses skip lists to exclude these from validation. Each model package has its own defaults.

In [10]:
print("=== Skip Lists for ODM 1.3.2 ===")
print(f"Skipped attributes: {checker.skip_attr}")
print(f"Skipped elements:   {checker.skip_elem}")

=== Skip Lists for ODM 1.3.2 ===
Skipped attributes: ['FileOID', 'PriorFileOID']
Skipped elements:   ['ODM']


You can extend the skip lists when creating a checker if your document has OID patterns that should be excluded:

In [11]:
# Add extra skips via the factory function parameters
custom_checker = create_oid_checker(
    "odm_1_3_2",
    extra_skip_attrs=["SomeCustomOID"],
    extra_skip_elems=["MyCustomElement"],
)

print(f"Extended skip_attr: {custom_checker.skip_attr}")
print(f"Extended skip_elem: {custom_checker.skip_elem}")

Extended skip_attr: ['FileOID', 'PriorFileOID', 'SomeCustomOID']
Extended skip_elem: ['ODM', 'MyCustomElement']


## 5. Cross-Model Support

The same `create_oid_checker()` API works across all supported model packages. Each model has its own set of OID definitions, reference attributes, and skip lists; all discovered automatically from the model classes.

Let's compare what the checker discovers for Define-XML 2.1 versus ODM 1.3.2. Define-XML extends ODM with additional elements like `ValueListDef`, `WhereClauseDef`, `CommentDef`, and `Standard`.

In [12]:
# Compare checkers for different models
checker_odm = create_oid_checker("odm_1_3_2")
checker_define = create_oid_checker("define_2_1")

print("=== ODM 1.3.2 ===")
print(f"  OID definition classes: {len(checker_odm.oid_defs)}")
print(f"  Ref -> Def mappings:    {len(checker_odm.ref_def)}")
print(f"  Skip attrs:             {checker_odm.skip_attr}")
print(f"  Skip elems:             {checker_odm.skip_elem}")

print(f"\n=== Define-XML 2.1 ===")
print(f"  OID definition classes: {len(checker_define.oid_defs)}")
print(f"  Ref -> Def mappings:    {len(checker_define.ref_def)}")
print(f"  Skip attrs:             {checker_define.skip_attr}")
print(f"  Skip elems:             {checker_define.skip_elem}")

# Show what Define-XML adds beyond ODM
odm_defs = set(checker_odm.oid_defs)
define_defs = set(checker_define.oid_defs)
define_only = define_defs - odm_defs
if define_only:
    print(f"\n  Define-XML specific definition classes: {sorted(define_only)}")

odm_refs = set(checker_odm.ref_def.keys())
define_refs = set(checker_define.ref_def.keys())
define_only_refs = define_refs - odm_refs
if define_only_refs:
    print(f"  Define-XML specific ref attributes:     {sorted(define_only_refs)}")

=== ODM 1.3.2 ===
  OID definition classes: 15
  Ref -> Def mappings:    15
  Skip attrs:             ['FileOID', 'PriorFileOID']
  Skip elems:             ['ODM']

=== Define-XML 2.1 ===
  OID definition classes: 11
  Ref -> Def mappings:    10
  Skip attrs:             ['FileOID', 'PriorFileOID', 'StudyOID', 'MetaDataVersionOID', 'ItemGroupOID']
  Skip elems:             ['ODM', 'Study', 'MetaDataVersion', 'ItemGroupDef']

  Define-XML specific definition classes: ['CommentDef', 'Standard', 'ValueListDef', 'WhereClauseDef', 'leaf']
  Define-XML specific ref attributes:     ['ArchiveLocationID', 'CommentOID', 'StandardOID', 'ValueListOID', 'WhereClauseOID', 'leafID']


### Using the Checker with a Define-XML Document

The checker works the same way with Define-XML. Here's an example with a programmatically built Define-XML 2.1 MetaDataVersion that includes Define-specific elements like `ValueListDef`, `WhereClauseDef`, and `CommentDef`.

> **Use the model package's own classes.** Build Define-XML 2.1 elements with `DEFINE.*` classes, not `ODM.*`. The Define-XML-specific elements (`CommentDef`, `Origin`, `ValueListDef`) bind the `define_2_1` `Description`/`TranslatedText` subclasses, so assigning an `odm_1_3_2` `ODM.Description()` to e.g. `CommentDef.Description` raises `OdmlibTypeError` (strict type checking). Mixing model packages is not supported outside of `permissive()` mode.

In [13]:
# Build a Define-XML 2.1 MetaDataVersion with interconnected elements.
# Use the DEFINE.* classes throughout: Define-XML-specific elements
# (CommentDef, Origin, ValueListDef) bind the define_2_1 Description/
# TranslatedText subclasses, so an odm_1_3_2 ODM.Description() is rejected.
wc = DEFINE.WhereClauseDef(OID="WC.DM.SEX")
vl = DEFINE.ValueListDef(OID="VL.DM.SEX")

# The ValueListDef contains an ItemRef that references both the ItemDef and WhereClauseDef
vl_item_ref = DEFINE.ItemRef(ItemOID="IT.DM.SEX", Mandatory="No")
vl_item_ref.WhereClauseRef = [DEFINE.WhereClauseRef(WhereClauseOID="WC.DM.SEX")]
vl.ItemRef = [vl_item_ref]

# ItemDef references a CodeList and a ValueList
item_def = DEFINE.ItemDef(OID="IT.DM.SEX", Name="Sex", DataType="text", Length=1)
item_def.CodeListRef = DEFINE.CodeListRef(CodeListOID="CL.SEX")
item_def.ValueListRef = DEFINE.ValueListRef(ValueListOID="VL.DM.SEX")

# A comment attached via CommentOID
comment = DEFINE.CommentDef(OID="COM.DM.001")
comment.Description = DEFINE.Description()
comment.Description.TranslatedText = [DEFINE.TranslatedText(_content="Demographics comment", lang="en")]

codelist = DEFINE.CodeList(OID="CL.SEX", Name="Sex", DataType="text", SASFormatName="SEX")

define_mdv = DEFINE.MetaDataVersion(
    OID="MDV.DEFINE", Name="Define Test", Description="OID check demo", DefineVersion="2.1.0",
)
define_mdv.ValueListDef = [vl]
define_mdv.WhereClauseDef = [wc]
define_mdv.ItemDef = [item_def]
define_mdv.CodeList = [codelist]
define_mdv.CommentDef = [comment]

# Validate with a Define-XML 2.1 checker
checker_d21 = create_oid_checker("define_2_1")
result = define_mdv.verify_oids(checker_d21)
print(f"Define-XML 2.1 OID validation passed: {result}")

print(f"\nOID definitions found: {len(checker_d21.oid)}")
for oid, elem_type in sorted(checker_d21.oid.items()):
    print(f"  {oid:20s} -> {elem_type}")

print(f"\nReferences validated:")
for attr, oid_set in sorted(checker_d21.oid_ref.items()):
    if oid_set:
        print(f"  {attr:25s} -> {sorted(oid_set)}")

Define-XML 2.1 OID validation passed: True

OID definitions found: 5
  CL.SEX               -> CodeList
  COM.DM.001           -> CommentDef
  IT.DM.SEX            -> ItemDef
  VL.DM.SEX            -> ValueListDef
  WC.DM.SEX            -> WhereClauseDef

References validated:
  CodeListOID               -> ['CL.SEX']
  ItemOID                   -> ['IT.DM.SEX']
  ValueListOID              -> ['VL.DM.SEX']
  WhereClauseOID            -> ['WC.DM.SEX']


### Supported Model Packages

`create_oid_checker()` supports these model packages out of the box:

| Model Package | Description | Key OID Types |
|---------------|-------------|---------------|
| `"odm_1_3_2"` | ODM v1.3.2 | StudyEventDef, FormDef, ItemGroupDef, ItemDef, CodeList, ConditionDef, etc. |
| `"define_2_0"` | Define-XML v2.0 | Adds ValueListDef, WhereClauseDef, CommentDef, leaf |
| `"define_2_1"` | Define-XML v2.1 | Adds Standard to Define 2.0 elements |
| `"odm_2_0"` | ODM v2.0 | Adds WorkflowDef, Arm, Epoch, StudyEventGroupDef, Transition |
| `"arm_1_0"` | ARM v1.0 | Analysis Results Metadata (extends Define 2.1) |

## 6. Migration from the Legacy API

Prior to v0.2.0, each model package had a manually-coded `OIDRef` class in `rules/oid_ref.py`. In v0.2.0, these classes are **deprecated** and will be removed in v0.3.0.

The migration is a one-line change:

In [14]:
import warnings

# BEFORE (deprecated) — issues OdmlibDeprecationWarning
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    from odmlib.define_2_1.rules.oid_ref import OIDRef as LegacyOIDRef
    legacy_checker = LegacyOIDRef()

print("Legacy OIDRef instantiation:")
for w in caught:
    print(f"  Warning: {w.message}")
    print(f"  Category: {w.category.__name__}")

# AFTER (recommended) — no warnings, auto-discovers mappings
dynamic_checker = create_oid_checker("define_2_1")
print(f"\nDynamic checker created: {len(dynamic_checker.ref_def)} ref/def mappings discovered")

Legacy OIDRef instantiation:
  Category: OdmlibDeprecationWarning

Dynamic checker created: 10 ref/def mappings discovered


### Why the Dynamic Approach Is Better

The legacy manual approach required maintaining three hand-coded dictionaries per model package:

```python
# Legacy: hard-coded in each rules/oid_ref.py (150+ lines per model)
class OIDRef:
    def _init_oid_ref(self):
        self.oid_ref["ItemOID"] = set()
        self.oid_ref["MethodOID"] = set()
        self.oid_ref["CodeListOID"] = set()
        # ... every reference attribute listed manually

    def _init_ref_def(self):
        self.ref_def["ItemOID"] = "ItemDef"
        self.ref_def["MethodOID"] = "MethodDef"
        # ... every mapping listed manually
```

The dynamic approach discovers everything automatically from the model class definitions, so it:

- **Stays in sync**: if a new element or reference attribute is added to the model, the checker picks it up automatically
- **Works across all models**: one implementation handles ODM 1.3.2, Define-XML 2.0/2.1, ODM 2.0, and ARM 1.0
- **Eliminates hand-coding bugs**: the legacy code had issues like trailing spaces in attribute names that the dynamic approach cannot have
- **Is extensible**: custom models are supported via `extra_skip_attrs` and `extra_skip_elems`

### Migration Checklist

| Old (pre-v0.2.0) | New (v0.2.0+) |
|---|---|
| `from odmlib.odm_1_3_2.rules.oid_ref import OIDRef` | `from odmlib import create_oid_checker` |
| `checker = OIDRef()` | `checker = create_oid_checker("odm_1_3_2")` |
| `from odmlib.define_2_1.rules.oid_ref import OIDRef` | `from odmlib import create_oid_checker` |
| `checker = OIDRef()` | `checker = create_oid_checker("define_2_1")` |

The rest of the code is unchanged, `verify_oids(checker)` and `unreferenced_oids(checker)` work identically with both the legacy and dynamic checkers.

## Summary

The dynamic OID ref/def checker in odmlib v0.2.0 provides a reliable, low-maintenance way to validate OID integrity in ODM documents:

- **`create_oid_checker(model_package)`** creates a checker by introspecting the model classes, no hand-coded mappings needed
- **`odm.verify_oids(checker)`** traverses the document tree, checks OID uniqueness, validates that every reference points to an existing definition of the correct type, and raises `OdmlibOIDError` on the first violation
- **`odm.unreferenced_oids(checker)`** finds orphaned definitions that are defined but never referenced
- **Structured errors**: `OdmlibOIDError` carries `attribute`, `hint`, and other context for programmatic reporting
- **Cross-model**: one API covers ODM 1.3.2, Define-XML 2.0/2.1, ODM 2.0, and ARM 1.0
- **Extensible**: `extra_skip_attrs` and `extra_skip_elems` let you customize validation for non-standard OID patterns
- **Backward compatible**: `OdmlibOIDError` inherits from `ValueError` through v0.2.x, so existing handlers still work